# Исследование игровой индустрии 2000-2013: подготовка и предобработка данных

- Автор: Павлович Арсений Владимирович
- Дата: 28.06.2026

### Цели и задачи проекта

**Цель:** подготовить корректный и очищенный срез исторических данных о продажах 
и оценках видеоигр за 2000-2013 гг., который после можно использовать для анализа 
игровой индустрии (для статьи-исследования).

**Задачи:**
- Загрузить данные и познакомиться с ними.
- Проверить данные на ошибки (названия столбцов, типы данных, пропуски, явные и неявные дубликаты) и провести предобработку.
- Отфильтровать данные по периоду 2000-2013 гг.
- Категоризировать игры по оценкам пользователей и критиков, выделить топ-7 платформ.
- Написать итоговые выводы по результатам проведённых мероприятий.

### Описание данных

Данные в файле `/datasets/new_games.csv` содержат информацию о продажах игр 
разных жанров и платформ, а также пользовательские и экспертные оценки игр.

| Столбец | Описание |
|---|---|
| `Name` | название игры |
| `Platform` | название платформы |
| `Year of Release` | год выпуска игры |
| `Genre` | жанр игры |
| `NA sales` | продажи в Северной Америке (млн проданных копий) |
| `EU sales` | продажи в Европе (млн проданных копий) |
| `JP sales` | продажи в Японии (млн проданных копий) |
| `Other sales` | продажи в других странах (млн проданных копий) |
| `Critic Score` | оценка критиков (от 0 до 100) |
| `User Score` | оценка пользователей (от 0 до 10) |
| `Rating` | рейтинг организации ESRB — присваивает игре подходящую возрастную категорию |

### Содержимое проекта

---

- [Загрузить данные и познакомиться с ними.](#target_1)
- [Проверить данные на ошибки (названия столбцов, типы данных, пропуски, явные и неявные дубликаты) и провести предобработку.](#target_2)
- [Отфильтровать данные по периоду 2000-2013 гг.](#target_3)
- [Категоризировать игры по оценкам пользователей и критиков, выделить топ-7 платформ.](#target_4)
- [Написать итоговые выводы по результатам проведённых мероприятий.](#target_5)

## 1. Загрузка данных и знакомство с ними <a id="target_1"></a>

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('https://code.s3.yandex.net/datasets/new_games.csv')

In [3]:
df.head()

,Name,Platform,Year of Release,Genre,NA sales,EU sales,JP sales,Other sales,Critic Score,User Score,Rating
0,Wii Sports,Wii,2006.0,Sports,41.36,28.96,3.77,8.45,76.0,8,E
1,Super Mario Bros.,NES,1985.0,Platform,29.08,3.58,6.81,0.77,NaN,NaN,NaN
2,Mario Kart Wii,Wii,2008.0,Racing,15.68,12.76,3.79,3.29,82.0,8.3,E
3,Wii Sports Resort,Wii,2009.0,Sports,15.61,10.93,3.28,2.95,80.0,8,E
4,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,11.27,8.89,10.22,1.00,NaN,NaN,NaN


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 16956 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Name             16954 non-null  str    
 1   Platform         16956 non-null  str    
 2   Year of Release  16681 non-null  float64
 3   Genre            16954 non-null  str    
 4   NA sales         16956 non-null  float64
 5   EU sales         16956 non-null  str    
 6   JP sales         16956 non-null  str    
 7   Other sales      16956 non-null  float64
 8   Critic Score     8242 non-null   float64
 9   User Score       10152 non-null  str    
 10  Rating           10085 non-null  str    
dtypes: float64(4), str(7)
memory usage: 1.4 MB


#### Вывод по полученным данным:

- Данные содержат **16 956 строк и 11 столбцов**, что соответствует описанию.
- В данных присутствует проблема в столбцах с продажами: 2 из 4 столбцов используют тип данных `str`. Конкретно `EU sales` и `JP sales` имеют тип `str`, тогда как `NA sales` и `Other sales` - `float64`. Для количества проданных копий уместен тип `float64`, так как значения дробные. Pandas прочитал числовой столбец как текст — скорее всего, в столбцах содержатся строковые значения.
- В столбце `User Score`, который является оценкой, также используется тип данных `str`, в то время как в схожем столбце `Critic Score` используется тип `float64`, что является правильным. Также в обоих столбцах присутствует много пропусков: в `Critic Score` из `16956` заполнено `8242`, а в `User Score` — `10152`.
- В столбце `Year of Release` есть проблема с типом данных: год не может быть вещественным числом. Причина — пропуски в этом столбце (из `16956` заполнено `16681`). Столбец с пропусками pandas не может держать как `int`, поэтому он стал `float`. Требует обработки после устранения пропусков.
- Также пропуски присутствуют в столбцах: `Rating` (из `16956` заполнено `10085`), а также `Name` и `Genre` (из `16956` заполнено `16954`, то есть не хватает по две строки).
- Названия столбцов записаны не в едином стиле — с заглавных букв и с пробелами (`Year of Release`, `NA sales`), что неудобно для обращения к ним в коде. Их следует привести к стилю snake_case.

---
## 2.  Проверка ошибок в данных и их предобработка <a id="target_2"></a>
### 2.1. Названия, или метки, столбцов датафрейма

In [5]:
print(df.columns)

Index(['Name', 'Platform', 'Year of Release', 'Genre', 'NA sales', 'EU sales',
       'JP sales', 'Other sales', 'Critic Score', 'User Score', 'Rating'],
      dtype='str')


In [6]:
df.columns = df.columns.str.lower().str.replace(' ', '_')

In [7]:
print(df.columns)

Index(['name', 'platform', 'year_of_release', 'genre', 'na_sales', 'eu_sales',
       'jp_sales', 'other_sales', 'critic_score', 'user_score', 'rating'],
      dtype='str')


**Вывод:** Названия столбцов приведены к стилю snake_case - нижний регистр, пробелы заменены на подчёркивания.

### 2.2. Типы данных

- Если встречаются некорректные типы данных, предположите их причины.
- При необходимости проведите преобразование типов данных. Помните, что столбцы с числовыми данными и пропусками нельзя преобразовать к типу `int64`. Сначала вам понадобится обработать пропуски, а затем преобразовать типы данных.

In [8]:
unique_values = df['user_score'].unique()

print(unique_values)

<StringArray>
[  '8',   nan, '8.3', '8.5', '6.6', '8.4', '8.6', '7.7', '6.3', '7.4', '8.2',
   '9', '7.9', '8.1', '8.7', '7.1', '3.4', '5.3', '4.8', '3.2', '8.9', '6.4',
 '7.8', '7.5', '2.6', '7.2', '9.2',   '7', '7.3', '4.3', '7.6', '5.7',   '5',
 '9.1', '6.5', 'tbd', '8.8', '6.9', '9.4', '6.8', '6.1', '6.7', '5.4',   '4',
 '4.9', '4.5', '9.3', '6.2', '4.2',   '6', '3.7', '4.1', '5.8', '5.6', '5.5',
 '4.4', '4.6', '5.9', '3.9', '3.1', '2.9', '5.2', '3.3', '4.7', '5.1', '3.5',
 '2.5', '1.9',   '3', '2.7', '2.2',   '2', '9.5', '2.1', '3.6', '2.8', '1.8',
 '3.8',   '0', '1.6', '9.6', '2.4', '1.7', '1.1', '0.3', '1.5', '0.7', '1.2',
 '2.3', '0.5', '1.3', '0.2', '0.6', '1.4', '0.9',   '1', '9.7']
Length: 97, dtype: str


-  среди значений `user_score` встречается текстовая заглушка `tbd` 
(«to be determined» – оценка ещё не определена). Именно из-за неё pandas прочитал 
весь столбец как `str`: в одном столбце не могут одновременно быть числа и текст. 
По смыслу `tbd` – это отсутствие данных, поэтому заменим её на `NaN` 
через `pd.to_numeric(errors='coerce')` и приведём столбец к числовому типу.

In [9]:
df['user_score'] = pd.to_numeric(df['user_score'], errors = 'coerce')

Аналогично проверяем `eu_sales` и `jp_sales`. Цикл выводит нечисловые значения 
(через сравнение исходных данных с результатом конвертации), затем приводит столбцы 
к числовому типу. Проверка показывает, что единственное «постороннее» значение – 
заглушка `unknown`, поэтому замена на `NaN` безопасна.

In [10]:
for col in ['eu_sales', 'jp_sales']:
    # пробуем превратить в число; что не вышло — станет NaN
    converted = pd.to_numeric(df[col], errors = 'coerce')
    # ищем строки, где конвертация дала NaN, но исходное значение НЕ было пустым
    bad_values = df[col][converted.isna() & df[col].notna()]
    df[col] = pd.to_numeric(df[col], errors = 'coerce')
    print(f'--- {col} ---')
    print(bad_values.unique())
    print()

--- eu_sales ---
<StringArray>
['unknown']
Length: 1, dtype: str

--- jp_sales ---
<StringArray>
['unknown']
Length: 1, dtype: str



In [11]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 16956 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   name             16954 non-null  str    
 1   platform         16956 non-null  str    
 2   year_of_release  16681 non-null  float64
 3   genre            16954 non-null  str    
 4   na_sales         16956 non-null  float64
 5   eu_sales         16950 non-null  float64
 6   jp_sales         16952 non-null  float64
 7   other_sales      16956 non-null  float64
 8   critic_score     8242 non-null   float64
 9   user_score       7688 non-null   float64
 10  rating           10085 non-null  str    
dtypes: float64(7), str(4)
memory usage: 1.4 MB


**Вывод:** в столбцах `eu_sales`, `jp_sales` и `user_score` встречаются строковые 
значения-заглушки (`unknown`, `tbd`), обозначающие отсутствие данных. Проверка показала, 
что других нечисловых значений нет, поэтому замена их на `NaN` через `errors = 'coerce'` 
безопасна – настоящие числа при этом не теряются. После замены столбцы приведены к типу `float64`. Также есть `year_of_release`, который нужно перевести в `int`, но в нем сейчас есть пропуски, поэтому к этому вернемся позже.

### 2.3. Наличие пропусков в данных

In [12]:
missing_info = pd.DataFrame({'Пропуски': df.isna().sum(), 'Процент (%)': (df.isna().mean() * 100).round(2)})

print(missing_info)

                 Пропуски  Процент (%)
name                    2         0.01
platform                0         0.00
year_of_release       275         1.62
genre                   2         0.01
na_sales                0         0.00
eu_sales                6         0.04
jp_sales                4         0.02
other_sales             0         0.00
critic_score         8714        51.39
user_score           9268        54.66
rating               6871        40.52


**Что можно сказать исходя из результатов количества пропусков в каждом столбце:** пропуски в данных можно разделить на группы по характеру и объёму:

| Столбцы | Доля пропусков | Действие | Обоснование |
|---|---|---|---|
| `critic_score`, `user_score`, `rating` | 40–55% | Оставить как `NaN` | Пропусков почти половина. Удаление строк уничтожит датасет и сделает анализ продаж нерепрезентативным. Заполнение средним выдумает данные и исказит связь «оценка ↔ продажи». Честный `NaN` – это само по себе информация («оценка неизвестна»), и pandas игнорирует его при расчётах. |
| `name`, `genre` | <0.1% (по 2 строки) | Удалить строки | Пропусков очень мало, на анализ не влияет. Восстановить название или жанр игры невозможно, а строка без названия будет битой записью. |
| `year_of_release` | 1.62% (275 строк) | Удалить строки | Год – ключевой признак для фильтрации периода 2000–2013. Восстановить его нельзя, а игры без года не смогут попасть в целевой срез, поэтому бесполезны для анализа. Доля пропусков невелика. |
| `eu_sales`, `jp_sales` | <0.05% (6 и 4 строки) | Заполнить средним по платформе и году | Продажи зависят от платформы (размер аудитории) и года (стадия рынка). Среднее внутри группы «та же платформа + тот же год» даёт реалистичную оценку по похожим рыночным условиям, в отличие от общего среднего по столбцу. |

Возможные причины пропусков в оценках: отсутствие оценок у старых игр (до эпохи Metacritic), нишевых проектов и переизданий; рейтинг ESRB отсутствует у игр, не выходивших в США.

- Обработайте пропущенные значения. Для каждого случая вы можете выбрать оптимальный, на ваш взгляд, вариант: заменить на определённое значение, оставить как есть или удалить.
- Если вы решите заменить пропуски на значение-индикатор, то убедитесь, что предложенное значение не может быть использовано в данных.
- Если вы нашли пропуски в данных с количеством проданных копий игры в том или ином регионе, их можно заменить на среднее значение в зависимости от названия платформы и года выхода игры.

In [13]:
rows_before = df.shape[0]

df = df.dropna(subset=['name', 'genre', 'year_of_release'])  # удаляем строки с пропусками в этих столбцах

rows_after = df.shape[0]

print('Удалено строк:', rows_before - rows_after)

Удалено строк: 277


Удалены 277 строк с пропусками в `name`, `genre` и `year_of_release`. 
Эти строки нельзя восстановить, а без года игра не попадёт в целевой срез 2000–2013, 
поэтому удаление оправдано.

Заполняем пропуски в `eu_sales` и `jp_sales` средним значением продаж по группе 
«платформа + год выпуска». Сначала считаем средние по группам, затем функцией 
подставляем их только в пропущенные значения (непропущенные остаются без изменений).

In [14]:
# Считаем среднее по группам

mean_eu = df.groupby(['platform', 'year_of_release'])['eu_sales'].mean()

In [15]:
def fill_eu(row):
    if pd.isna(row['eu_sales']):                                 # если пропуск
        return mean_eu[row['platform'], row['year_of_release']]  # берём среднее группы
    else:
        return row['eu_sales']                                   # иначе оставляем как было

In [16]:
df['eu_sales'] = df.apply(fill_eu, axis=1)

In [17]:
# Тоже самое, но с jp_sales

mean_jp = df.groupby(['platform', 'year_of_release'])['jp_sales'].mean()

In [18]:
def fill_jp(row):
    if pd.isna(row['jp_sales']):                                 # если пропуск
        return mean_jp[row['platform'], row['year_of_release']]  # берём среднее группы
    else:
        return row['jp_sales']                                   # иначе оставляем как было

In [19]:
df['jp_sales'] = df.apply(fill_jp, axis=1)

In [20]:
print(df[['eu_sales', 'jp_sales']].isna().sum())

eu_sales    0
jp_sales    0
dtype: int64


In [21]:
# Сделал проверку describe(), чтобы успокоить себя, что заполненные значения адекватны – нет отрицательных продаж и аномальных выбросов.

print(df[['eu_sales', 'jp_sales']].describe())

           eu_sales      jp_sales
count  16679.000000  16679.000000
mean       0.144987      0.078031
std        0.503605      0.309366
min        0.000000      0.000000
25%        0.000000      0.000000
50%        0.020000      0.000000
75%        0.110000      0.040000
max       28.960000     10.220000


**Вывод:** пропущенные значения обработаны согласно плану выше:
- Удалены 277 строк с пропусками в `name`, `genre` и `year_of_release` (битые записи / отсутствие года, восстановить нельзя).
- Пропуски в `eu_sales` и `jp_sales` заполнены средним значением продаж по группе «платформа + год выпуска». После заполнения пропусков в этих столбцах не осталось (0).
- Пропуски в `critic_score`, `user_score` и `rating` оставлены как `NaN`, так как их доля велика (40-55%), а заполнение исказило бы анализ.

### 2.4. Явные и неявные дубликаты в данных

In [22]:
for col in ['platform', 'genre', 'rating', 'year_of_release']:
    print(f'--- {col} ---')
    print(df[col].unique())
    print()

--- platform ---
<StringArray>
[ 'Wii',  'NES',   'GB',   'DS', 'X360',  'PS3',  'PS2', 'SNES',  'GBA',
  'PS4',  '3DS',  'N64',   'PS',   'XB',   'PC', '2600',  'PSP', 'XOne',
 'WiiU',   'GC',  'GEN',   'DC',  'PSV',  'SAT',  'SCD',   'WS',   'NG',
 'TG16',  '3DO',   'GG', 'PCFX']
Length: 31, dtype: str

--- genre ---
<StringArray>
[      'Sports',     'Platform',       'Racing', 'Role-Playing',
       'Puzzle',         'Misc',      'Shooter',   'Simulation',
       'Action',     'Fighting',    'Adventure',     'Strategy',
         'MISC', 'ROLE-PLAYING',       'RACING',       'ACTION',
      'SHOOTER',     'FIGHTING',       'SPORTS',     'PLATFORM',
    'ADVENTURE',   'SIMULATION',       'PUZZLE',     'STRATEGY']
Length: 24, dtype: str

--- rating ---
<StringArray>
['E', nan, 'M', 'T', 'E10+', 'K-A', 'AO', 'EC', 'RP']
Length: 9, dtype: str

--- year_of_release ---
[2006. 1985. 2008. 2009. 1996. 1989. 1984. 2005. 1999. 2007. 2010. 2013.
 2004. 1990. 1988. 2002. 2001. 2011. 1998. 2015.

In [23]:
# Нашел, что в genre находятся неявные дубликаты, почти каждый жанр продублирован в верхнем регистре

df['genre'] = df['genre'].str.lower()

- После того как нормализуете данные и устраните неявные дубликаты, проверьте наличие явных дубликатов в данных.

In [24]:
print('Явных дубликатов:', df.duplicated().sum())

Явных дубликатов: 235


In [25]:
# Удаляем дубликаты, которые мы выявили.

rows_before = df.shape[0]
df = df.drop_duplicates().reset_index(drop=True) # reset_index, чтобы избавиться от дырявых индексов, которые потом могут повлиять
rows_after = df.shape[0]
print('Удалено дубликатов:', rows_before - rows_after)

Удалено дубликатов: 235


In [26]:
# Возврат к отложенной задаче из 2.2: пропусков в year_of_release больше нет, 
# приводим столбец к целочисленному типу
df['year_of_release'] = df['year_of_release'].astype('int16')

# Проверка
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 16444 entries, 0 to 16443
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   name             16444 non-null  str    
 1   platform         16444 non-null  str    
 2   year_of_release  16444 non-null  int16  
 3   genre            16444 non-null  str    
 4   na_sales         16444 non-null  float64
 5   eu_sales         16444 non-null  float64
 6   jp_sales         16444 non-null  float64
 7   other_sales      16444 non-null  float64
 8   critic_score     7983 non-null   float64
 9   user_score       7463 non-null   float64
 10  rating           9768 non-null   str    
dtypes: float64(6), int16(1), str(4)
memory usage: 1.3 MB


**Небольшая ремарка:** 
- При работе с большими объёмами данных год можно хранить в более компактном типе (например, int16) для экономии памяти, но при текущем размере датасета (16 тыс. строк) это несущественно, но я все равно это сделаю, потому что помню, как это описывалось в теории! Не знаю, правильно так делать или нет по сути ~96 кб.

**Вывод по дубликатам:**

- В `genre` обнаружены неявные дубликаты – жанры в двух регистрах (24 уникальных значения). После нормализации к нижнему регистру стало 12.
- В `platform` и `rating` неявных дубликатов нет – это корректные коды платформ и рейтингов ESRB.
- Явных дубликатов найдено **235**, удалены, индекс переиндексирован.

- В процессе подготовки данных вы могли что-либо удалять, например строки с пропусками или ошибками, дубликаты и прочее. В этом случае посчитайте количество удалённых строк в абсолютном и относительном значениях.

In [27]:
# Считаем общее количество строк, удалённых за всю предобработку
# (пропуски в name/genre/year + явные дубликаты)

rows_start = 16956  # исходное число строк (из самого первого info())
rows_now = df.shape[0]
deleted = rows_start - rows_now

print('Было строк:', rows_start)
print('Стало строк:', rows_now)
print('Удалено строк:', deleted)
print('Доля удалённых:', round(deleted / rows_start * 100, 2), '%')

Было строк: 16956
Стало строк: 16444
Удалено строк: 512
Доля удалённых: 3.02 %


- После проведения предобработки данных напишите общий промежуточный вывод.

**Вывод по предобработке:**

- **Названия столбцов** приведены к единому формату snake_case (нижний регистр, пробелы заменены на подчёркивания).

- **Типы данных.** В столбцах `user_score`, `eu_sales`, `jp_sales` встречались строковые заглушки (`tbd`, `unknown`), из-за которых pandas прочитал их как `str`. Заменили заглушки на `NaN` через `pd.to_numeric(errors='coerce')` и привели столбцы к `float64`. После удаления пропусков привели `year_of_release` к типу `int16`.

- **Пропуски** обработаны тремя способами в зависимости от их характера: удалены строки с пропусками в `name`, `genre`, `year_of_release` (восстановить нельзя); пропуски в `eu_sales` и `jp_sales` заполнены средним по группе «платформа + год»; пропуски в `critic_score`, `user_score`, `rating` оставлены как `NaN`, так как их доля велика (40-55%) и заполнение исказило бы анализ.

- **Дубликаты.** В `genre` найдены неявные дубликаты (жанры в разных регистрах) – 24 уникальных значения, после нормализации к нижнему регистру осталось 12. Также найдено и удалено 235 явных дубликатов.

- **Итог.** Из 16956 строк осталось 16444 – удалено 512 строк (около 3%). Потеря небольшая, данные очищены и готовы к дальнейшему анализу.

---
## 3. Фильтрация данных <a id="target_3"></a>

In [28]:
df_actual = df[(df['year_of_release'] >= 2000) & (df['year_of_release'] <= 2013)]

In [29]:
print('Строк в срезе:', df_actual.shape[0])

Строк в срезе: 12781


In [30]:
# Убедимся, что ничего лишнего не залезло в результат:

print('Годы в срезе:', df_actual['year_of_release'].min(), '-', df_actual['year_of_release'].max())

Годы в срезе: 2000 - 2013


**Вывод:**

- Отфильтровали данные по периоду 2000-2013 (включительно) и сохранили срез в отдельный датафрейм `df_actual`.
- Из 16444 строк в срез вошло 12781 – отсеялись игры, выпущенные до 2000 и после 2013 года (3663 строки).
- Проверка по минимальному и максимальному году (2000 и 2013) подтверждает, что фильтр захватил корректный диапазон.

---
## 4. Категоризация данных <a id="target_4"></a>

In [31]:
def user_category(score):
    if 8 <= score <= 10:      # от 8 до 10 включительно
        return 'высокая'
    elif 3 <= score < 8:      # от 3 до 8 (без правой границы)
        return 'средняя'
    elif 0 <= score < 3:      # от 0 до 3 (без правой границы)
        return 'низкая'
    else:
        return 'без оценки'

In [32]:
df_actual['user_score_category'] = df_actual['user_score'].apply(user_category)

- Проверим результат сгруппировав данные по категориям и посчитаем кол-во игр в каждой категории:

In [33]:
print(df_actual['user_score_category'].value_counts())

user_score_category
без оценки    6298
средняя       4081
высокая       2286
низкая         116
Name: count, dtype: int64


In [34]:
def critic_category(score):
    if 80 <= score <= 100:
        return 'высокая'
    elif 30 <=  score < 80:
        return 'средняя'
    elif 0 <= score < 30:
        return 'низкая'
    else:
        return 'без оценки'

In [35]:
df_actual['critic_score_category'] = df_actual['critic_score'].apply(critic_category)

- Проверим результат сгруппировав данные по категориям и посчитаем кол-во игр в каждой категории:

In [36]:
print(df_actual['critic_score_category'].value_counts())

critic_score_category
без оценки    5612
средняя       5422
высокая       1692
низкая          55
Name: count, dtype: int64


Игры без оценки (NaN в `user_score`/`critic_score`) выделены в отдельную 
   категорию «без оценки», чтобы не терять эти строки и явно показать, 
   что оценка отсутствует.

- После категоризации данных проверьте результат: сгруппируйте данные по выделенным категориям и посчитайте количество игр в каждой категории.

- **Проверил сразу после применения функции** 

- Выделите топ-7 платформ по количеству игр, выпущенных за весь актуальный период.

In [37]:
top_7_platforms = df_actual['platform'].value_counts().head(7)

In [38]:
print(top_7_platforms)

platform
PS2     2127
DS      2120
Wii     1275
PSP     1180
X360    1121
PS3     1087
GBA      811
Name: count, dtype: int64


**Вывод:** топ-7 платформ по количеству выпущенных игр за 2000–2013: 
   PS2, DS, Wii, PSP, X360, PS3, GBA. Лидируют консоли Sony и Nintendo – 
   это соответствует расцвету этих платформ в указанный период.

---
## 5. Итоговый вывод <a id="target_5"></a>

**Итоговый вывод**
В рамках проекта были подготовлены исторические данные о продажах видеоигр для дальнейшего анализа игровой индустрии.
Исходный датасет содержал `16956` строк с рядом проблем: некорректные типы данных, пропуски и дубликаты. В ходе предобработки:
- названия столбцов приведены к стилю snake_case и нижнему регистру;
- исправлены типы данных (столбцы `user_score`, `eu_sales`, `jp_sales`, приведены к числовому типу, `year_of_release` – к int16 для экономии места);
- обработаны пропуски: часть строк удалена, продажи заполнены средним значением по группе «платформа + год выпуска», оценки оставлены как `NaN`;
- удалены неявные и явные дубликаты.
Всего удалено `512` строк (около `3%`), осталось `16444` строк.
**Срез данных:** для анализа отобран период `2000-2013` годов (включительно) и сохранён в отдельный датафрейм `df_actual` объёмом `12781` строка.
**Новые поля**, добавленные в данные:
- `user_score_category` – разделяет все игры по оценкам пользователей, где высокая оценка (от 8 до 10 включительно), средняя оценка (от 3 до 8, не включая правую границу интервала) и низкая оценка (от 0 до 3, не включая правую границу интервала) и без оценки (для пустых значений);
- `critic_score_category` – разделяет все игры по оценкам критиков, где высокая оценка (от 80 до 100 включительно), средняя оценка (от 30 до 80, не включая правую границу интервала) и низкая оценка (от 0 до 30, не включая правую границу интервала) и без оценки (для пустых значений).
Также определён **топ-7 платформ** по количеству выпущенных игр: PS2, DS, Wii, PSP, X360, PS3, GBA. Лидируют консоли Sony и Nintendo.
Данные очищены, структурированы и готовы для дальнейшего анализа.